# Crypto 5분봉 Data - FinRL 튜토리얼
이 노트북은 **암호화폐 5분봉 데이터**를 사용하여 강화학습 트레이딩을 위한 데이터를 준비합니다.

CCXT 라이브러리를 통해 Binance에서 데이터를 가져옵니다.

원본: Stock NeurIPS2018 Part 1. Data

# Part 1. Install Packages

In [ ]:
# 이미 가상환경에 설치되어 있다면 생략 가능
# !pip install ccxt finrl

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# CCXT를 사용한 암호화폐 데이터 수집
from finrl.meta.data_processors.processor_ccxt import CCXTEngineer
from finrl.meta.preprocessor.preprocessors import FeatureEngineer, data_split
from finrl.config import INDICATORS

# Part 2. Fetch data

[CCXT](https://github.com/ccxt/ccxt)는 100개 이상의 암호화폐 거래소를 지원하는 라이브러리입니다. FinRL의 CCXTEngineer를 사용하여 Binance에서 5분봉 데이터를 가져옵니다.

**OHLCV**: Data downloaded are in the form of OHLCV, corresponding to **open, high, low, close, volume,** respectively. OHLCV is important because they contain most of numerical information of a stock in time series. From OHLCV, traders can get further judgement and prediction like the momentum, people's interest, market trends, etc.

## 단일 코인 데이터 (BTC/USDT)

Bitcoin(BTC/USDT) 5분봉 데이터를 가져옵니다. 날짜 형식: "YYYYMMDD HH:MM:SS"

### CCXTEngineer 초기화

In [2]:
# CCXTEngineer 초기화
ccxt_eng = CCXTEngineer()

# 기간 설정 (최근 7일 데이터 - 5분봉은 데이터가 많으므로 짧은 기간 권장)
end_date = datetime.now()
start_date = end_date - timedelta(days=7)

START_DATE = start_date.strftime("%Y%m%d %H:%M:%S")
END_DATE = end_date.strftime("%Y%m%d %H:%M:%S")

print(f"시작: {START_DATE}")
print(f"종료: {END_DATE}")

시작: 20260117 22:04:12
종료: 20260124 22:04:12


In [3]:
# BTC/USDT 5분봉 데이터 가져오기
btc_df = ccxt_eng.data_fetch(
    start=START_DATE,
    end=END_DATE,
    pair_list=["BTC/USDT"],
    period="5m"  # 5분봉
)
print(f"데이터 shape: {btc_df.shape}")
btc_df.head()

Actual end time: 2026-01-24T22:00:00.000000000
데이터 shape: (1908, 5)


BTC/USDT                                        
                         open      high       low     close    volume
2026-01-18 07:05:00  95289.15  95299.99  95289.14  95289.92   9.69685
2026-01-18 07:10:00  95289.92  95289.93  95154.49  95188.00  30.58841
2026-01-18 07:15:00  95188.00  95238.92  95187.99  95238.91   8.79242
2026-01-18 07:20:00  95238.92  95238.92  95188.26  95188.27  10.79829
2026-01-18 07:25:00  95188.28  95197.21  95164.00  95175.01  18.29941

### 데이터 확인

5분봉 데이터는 하루에 288개의 캔들이 생성됩니다 (24시간 * 60분 / 5분 = 288)

In [4]:
# 데이터 통계
print(f"총 데이터 수: {len(btc_df)}")
print(f"시작 시간: {btc_df.index[0]}")
print(f"종료 시간: {btc_df.index[-1]}")

총 데이터 수: 1908
시작 시간: 2026-01-18 07:05:00
종료 시간: 2026-01-24 22:00:00


In [5]:
btc_df.tail()

BTC/USDT                                        
                         open      high       low     close    volume
2026-01-24 21:40:00  89520.00  89536.00  89519.99  89535.99   9.67476
2026-01-24 21:45:00  89536.00  89536.00  89523.13  89523.14  19.36286
2026-01-24 21:50:00  89523.13  89537.82  89523.13  89537.82  13.91854
2026-01-24 21:55:00  89537.81  89537.82  89537.81  89537.82  11.55687
2026-01-24 22:00:00  89537.82  89537.82  89537.81  89537.81   8.43744

## 다중 코인 데이터

In [6]:
# 트레이딩할 코인 리스트 정의
CRYPTO_PAIRS = [
    "BTC/USDT",   # 비트코인
    "ETH/USDT",   # 이더리움
    "BNB/USDT",   # 바이낸스 코인
    "XRP/USDT",   # 리플
    "SOL/USDT",   # 솔라나
]
print(f"트레이딩 대상 코인: {CRYPTO_PAIRS}")

트레이딩 대상 코인: ['BTC/USDT', 'ETH/USDT', 'BNB/USDT', 'XRP/USDT', 'SOL/USDT']


In [7]:
# 학습/테스트 기간 설정
# 5분봉 데이터는 양이 많으므로 30일 정도가 적당
end_date = datetime.now()
train_start = end_date - timedelta(days=30)  # 30일 전부터
train_end = end_date - timedelta(days=7)     # 7일 전까지 학습
trade_start = end_date - timedelta(days=7)   # 7일 전부터
trade_end = end_date                         # 현재까지 테스트

TRAIN_START = train_start.strftime("%Y%m%d %H:%M:%S")
TRAIN_END = train_end.strftime("%Y%m%d %H:%M:%S")
TRADE_START = trade_start.strftime("%Y%m%d %H:%M:%S")
TRADE_END = trade_end.strftime("%Y%m%d %H:%M:%S")

print(f"학습 기간: {TRAIN_START} ~ {TRAIN_END}")
print(f"테스트 기간: {TRADE_START} ~ {TRADE_END}")

학습 기간: 20251225 22:04:36 ~ 20260117 22:04:36
테스트 기간: 20260117 22:04:36 ~ 20260124 22:04:36


In [8]:
# 다중 코인 5분봉 데이터 가져오기 (전체 기간)
df_raw = ccxt_eng.data_fetch(
    start=TRAIN_START,
    end=TRADE_END,
    pair_list=CRYPTO_PAIRS,
    period="5m"
)
print(f"데이터 shape: {df_raw.shape}")

Actual end time: 2026-01-24T22:00:00.000000000
데이터 shape: (8532, 25)


In [9]:
df_raw.head()

BTC/USDT                                          \
                         open      high       low     close    volume   
2025-12-26 07:05:00  87915.09  87915.09  87792.67  87807.81  15.24484   
2025-12-26 07:10:00  87807.81  87871.06  87804.80  87855.60  52.63615   
2025-12-26 07:15:00  87855.60  87860.97  87827.21  87860.97   4.56669   
2025-12-26 07:20:00  87860.96  87860.97  87825.06  87825.07   6.03272   
2025-12-26 07:25:00  87825.07  87872.93  87822.02  87842.85  46.78312   

                    ETH/USDT                                       ...  \
                        open     high      low    close    volume  ...   
2025-12-26 07:05:00  2948.11  2948.12  2942.34  2942.70  606.1716  ...   
2025-12-26 07:10:00  2942.69  2943.19  2937.59  2940.26  923.7293  ...   
2025-12-26 07:15:00  2940.26  2943.45  2937.53  2943.45  435.0727  ...   
2025-12-26 07:20:00  2943.45  2943.45  2940.42  2940.98  206.8306  ...   
2025-12-26 07:25:00  2940.99  2941.59  2939.43  2939.44  322.0340  ...   

                    XRP/USDT                                   SOL/USDT  \
                        open    high     low   close    volume     open   
2025-12-26 07:05:00   1.8637  1.8637  1.8591  1.8606  388663.4   123.36   
2025-12-26 07:10:00   1.8605  1.8625  1.8601  1.8608  148910.0   123.05   
2025-12-26 07:15:00   1.8609  1.8624  1.8595  1.8616  110769.4   123.01   
2025-12-26 07:20:00   1.8616  1.8616  1.8596  1.8599  105662.7   123.08   
2025-12-26 07:25:00   1.8600  1.8614  1.8594  1.8611  106406.3   122.96   

                                                       
                       high     low   close    volume  
2025-12-26 07:05:00  123.36  123.02  123.05  3765.258  
2025-12-26 07:10:00  123.12  122.98  123.01  5817.009  
2025-12-26 07:15:00  123.10  122.90  123.07  1500.134  
2025-12-26 07:20:00  123.09  122.94  122.96  2219.301  
2025-12-26 07:25:00  123.07  122.96  122.97  1543.983  

[5 rows x 25 columns]

# Part 3: 기술적 지표 추가
강화학습 에이전트가 더 나은 결정을 내릴 수 있도록 기술적 지표를 추가합니다:
* **MACD**: 추세 추종 지표
* **RSI**: 과매수/과매도 지표
* **Bollinger Bands**: 변동성 지표
* **SMA**: 이동평균선

CCXTEngineer의 `add_technical_indicators` 메서드를 사용하여 기술적 지표를 추가합니다.

In [10]:
# 기술적 지표 리스트
TECH_INDICATORS = [
    "macd",
    "boll_ub",
    "boll_lb",
    "rsi_30",
    "dx_30",
    "close_30_sma",
    "close_60_sma",
]

# 기술적 지표 추가
df_with_indicators = ccxt_eng.add_technical_indicators(
    df=df_raw,
    pair_list=CRYPTO_PAIRS,
    tech_indicator_list=TECH_INDICATORS
)
print(f"지표 추가 후 shape: {df_with_indicators.shape}")

Succesfully add technical indicators
지표 추가 후 shape: (8532, 60)


In [11]:
# 결과 확인
df_with_indicators.head()

BTC/USDT                                          \
                         open      high       low     close    volume   
2025-12-26 07:05:00  87915.09  87915.09  87792.67  87807.81  15.24484   
2025-12-26 07:10:00  87807.81  87871.06  87804.80  87855.60  52.63615   
2025-12-26 07:15:00  87855.60  87860.97  87827.21  87860.97   4.56669   
2025-12-26 07:20:00  87860.96  87860.97  87825.06  87825.07   6.03272   
2025-12-26 07:25:00  87825.07  87872.93  87822.02  87842.85  46.78312   

                                                                       \
                         macd       boll_ub       boll_lb      rsi_30   
2025-12-26 07:05:00  0.000000           NaN           NaN         NaN   
2025-12-26 07:10:00  1.072212  87899.290266  87764.119734  100.000000   
2025-12-26 07:15:00  1.532734  87899.990372  87782.929628  100.000000   
2025-12-26 07:20:00  0.408469  87887.884789  87786.840211   58.133183   
2025-12-26 07:25:00  0.496468  87882.488018  87794.431982   65.527573   

                                 ... SOL/USDT                              \
                          dx_30  ...      low   close    volume      macd   
2025-12-26 07:05:00         NaN  ...   123.02  123.05  3765.258  0.000000   
2025-12-26 07:10:00         NaN  ...   122.98  123.01  5817.009 -0.000897   
2025-12-26 07:15:00         NaN  ...   122.90  123.07  1500.134  0.000717   
2025-12-26 07:20:00  100.000000  ...   122.94  122.96  2219.301 -0.002538   
2025-12-26 07:25:00   71.208238  ...   122.96  122.97  1543.983 -0.003855   

                                                                            \
                        boll_ub     boll_lb     rsi_30  dx_30 close_30_sma   
2025-12-26 07:05:00         NaN         NaN        NaN    NaN   123.050000   
2025-12-26 07:10:00  123.086569  122.973431   0.000000  100.0   123.030000   
2025-12-26 07:15:00  123.104434  122.982232  60.810811  100.0   123.043333   
2025-12-26 07:20:00  123.119625  122.925375  28.240641  100.0   123.022500   
2025-12-26 07:25:00  123.108333  122.915667  31.681811  100.0   123.012000   

                                  
                    close_60_sma  
2025-12-26 07:05:00   123.050000  
2025-12-26 07:10:00   123.030000  
2025-12-26 07:15:00   123.043333  
2025-12-26 07:20:00   123.022500  
2025-12-26 07:25:00   123.012000  

[5 rows x 60 columns]

In [12]:
# 컬럼 구조 확인
print("컬럼 구조:")
print(df_with_indicators.columns.tolist()[:20])  # 처음 20개만 출력

컬럼 구조:
[('BTC/USDT', 'open'), ('BTC/USDT', 'high'), ('BTC/USDT', 'low'), ('BTC/USDT', 'close'), ('BTC/USDT', 'volume'), ('BTC/USDT', 'macd'), ('BTC/USDT', 'boll_ub'), ('BTC/USDT', 'boll_lb'), ('BTC/USDT', 'rsi_30'), ('BTC/USDT', 'dx_30'), ('BTC/USDT', 'close_30_sma'), ('BTC/USDT', 'close_60_sma'), ('ETH/USDT', 'open'), ('ETH/USDT', 'high'), ('ETH/USDT', 'low'), ('ETH/USDT', 'close'), ('ETH/USDT', 'volume'), ('ETH/USDT', 'macd'), ('ETH/USDT', 'boll_ub'), ('ETH/USDT', 'boll_lb')]


# Part 4: 데이터 배열 변환 및 저장

### 강화학습용 배열로 변환
CCXTEngineer의 `df_to_ary` 메서드를 사용하여 가격 배열과 기술적 지표 배열로 변환합니다.

In [13]:
# 배열로 변환
price_array, tech_array, date_array = ccxt_eng.df_to_ary(
    df=df_with_indicators,
    pair_list=CRYPTO_PAIRS,
    tech_indicator_list=TECH_INDICATORS
)

print(f"가격 배열 shape: {price_array.shape}")  # (시간, 코인수)
print(f"기술지표 배열 shape: {tech_array.shape}")  # (시간, 코인수 * 지표수)
print(f"날짜 배열 shape: {date_array.shape}")

가격 배열 shape: (8529, 5)
기술지표 배열 shape: (8529, 35)
날짜 배열 shape: (8529,)


### 학습/테스트 데이터 분리

In [14]:
# 학습/테스트 분리 (80% 학습, 20% 테스트)
train_size = int(len(price_array) * 0.8)

train_price = price_array[:train_size]
train_tech = tech_array[:train_size]
train_date = date_array[:train_size]

test_price = price_array[train_size:]
test_tech = tech_array[train_size:]
test_date = date_array[train_size:]

print(f"학습 데이터: {len(train_price)} 샘플")
print(f"테스트 데이터: {len(test_price)} 샘플")

학습 데이터: 6823 샘플
테스트 데이터: 1706 샘플


In [15]:
# 데이터 저장 (numpy 배열로 저장)
np.savez(
    'crypto_5m_data.npz',
    train_price=train_price,
    train_tech=train_tech,
    train_date=train_date,
    test_price=test_price,
    test_tech=test_tech,
    test_date=test_date,
    crypto_pairs=CRYPTO_PAIRS,
    tech_indicators=TECH_INDICATORS
)

# DataFrame도 저장
df_with_indicators.to_csv('crypto_5m_data.csv')

print("저장 완료!")
print("- crypto_5m_data.npz: 강화학습용 배열 데이터")
print("- crypto_5m_data.csv: 원본 DataFrame")

저장 완료!
- crypto_5m_data.npz: 강화학습용 배열 데이터
- crypto_5m_data.csv: 원본 DataFrame
